# 05 — String-Level Disagreement Analysis

This notebook characterises how much services agree on translations and, when they disagree, what *kind* of disagreement it is; using only the output strings and external metadata (Wikipedia presence, source-term borrowing detectable in the string).

**Methodological note — no rationale keywords.**  
The classifier deliberately excludes the models' rationale text from the classification signal. This keeps the disagreement labels independent of the rationale analysis in notebook 07, avoiding circularity. The models' own reasoning is analysed separately as an independent axis in that notebook.

Five categories, ordered from most to least certain:

| Category | String signal |
|---|---|
| `COMPLETE_CONSENSUS` | All services produced the same string — no disagreement |
| `MEASUREMENT_ARTEFACT` | Disagreement collapses after normalisation (case / diacritics / script) |
| `PRODUCTIVE_DISAGREEMENT` | Multiple distinct outputs AND Wikipedia translation exists |
| `STRUCTURAL_ABSENCE` | At least one service borrowed the source term unadapted |
| `TRANSMOGRIFICATION` | Multiple distinct outputs, no Wikipedia anchor, no borrowing |

Input: `translated_terms/digital_humanities/evaluation/across_variant_detail.csv`  
Output: `translated_terms/digital_humanities/evaluation/disagreement_analysis.csv`
**Exclusion tier: Tier 2 — Translation Analysis** (see [docs/exclusion_strategy.md](../docs/exclusion_strategy.md))

Term-error flags (`has_mixed_script`, `has_placeholder_term`, `has_repetition_loop`, `has_extreme_term_length`, `has_unicode_escape`) are used to drop individual service terms before classification. Manual `exclude_translation=True` entries are also applied. `has_source_term` and `has_script_disagreement` are retained — borrowing and cross-service script divergence are classification inputs, not errors.


## 5.1 Setup & Run

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.enable('vegafusion')

sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import get_data_directory_path, read_csv_file
from scripts.exploration.explore_disagreements import (
    ClassifierConfig,
    CATEGORIES,
    run_disagreement_analysis,
    load_exclusions,
)

In [2]:
DATA_DIR         = get_data_directory_path()
TERM_SLUG        = 'digital_humanities'
EVAL_DIR         = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
DISAGR_CSV       = os.path.join(EVAL_DIR, 'disagreement_analysis.csv')
DISAGR_NO_KW_CSV = os.path.join(EVAL_DIR, 'disagreement_analysis_no_keywords.csv')
DISAGR_KW_CSV    = os.path.join(EVAL_DIR, 'disagreement_analysis_keywords.csv')

print(f'Eval dir       : {EVAL_DIR}')
print(f'Canonical CSV  : {DISAGR_CSV}')
print(f'No-keywords CSV: {DISAGR_NO_KW_CSV}')
print(f'Keywords CSV   : {DISAGR_KW_CSV}')
EXCL_PATH        = os.path.join(EVAL_DIR, 'manual_exclusions.csv')
FLAGS_PATH       = os.path.join(EVAL_DIR, 'quality_flags.csv')


Retrieving translation pipeline data directory path...

Eval dir       : /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation
Canonical CSV  : /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/disagreement_analysis.csv
No-keywords CSV: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/disagreement_analysis_no_keywords.csv
Keywords CSV   : /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/disagreement_analysis_keywords.csv


In [3]:
# ── Tier 2 exclusions: quality flags + manual exclusions ────────────────────
# Term-error flags are converted to the same (language × service) exclusion
# format that run_disagreement_analysis already understands.

quality_flags_df = read_csv_file(FLAGS_PATH)

# ── Language count sanity check (pre-exclusion) ─────────────────────────────
_n = quality_flags_df['language_code'].nunique()
print(f'Languages loaded (pre-exclusion): {_n} (expected 880)')
if _n != 880:
    print(f'  ⚠ Expected 880 — re-run the translation pipeline or check for missing/duplicate language_code rows.')
else:
    print(f'  ✓ Count matches expected 880.')

TERM_ERROR_COLS = {
    'has_mixed_script':        'mixed_script_services',
    'has_placeholder_term':    'placeholder_term_services',
    'has_repetition_loop':     'repetition_loop_services',
    'has_extreme_term_length': 'extreme_term_length_services',
    'has_unicode_escape':      'unicode_escape_services',
}

def _empty_entry():
    return {'term': False, 'rationale_minimal': False,
            'rationale_expert_persona': False,
            'rationale_native_rationale': False, 'rationale_judge': False}

tier2_excl = {}
for flag_col, svc_col in TERM_ERROR_COLS.items():
    for _, row in quality_flags_df[quality_flags_df[flag_col] == True].iterrows():
        lc = row['language_code']
        svcs = [s.strip() for s in str(row.get(svc_col, '')).split(';')
                if s.strip() and s.strip() != 'nan']
        for svc in svcs:
            tier2_excl.setdefault(lc, {}).setdefault(svc, _empty_entry())['term'] = True

if os.path.exists(EXCL_PATH):
    manual_excl = load_exclusions(EXCL_PATH)
    for lc, svc_dict in manual_excl.items():
        for svc, entry in svc_dict.items():
            existing = tier2_excl.setdefault(lc, {}).setdefault(svc, _empty_entry())
            for k, v in entry.items():
                existing[k] = existing.get(k, False) or v

n_term = sum(1 for svcs in tier2_excl.values() for e in svcs.values() if e['term'])
print(f'Tier 2: {n_term} (language × service) term exclusions')


Languages loaded (pre-exclusion): 880 (expected 880)
  ✓ Count matches expected 880.


  Loaded 275 exclusion entries across 235 languages from 
/Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evalu
ation/manual_exclusions.csv

Tier 2: 257 (language × service) term exclusions


In [4]:
# ── Exclusion summary (uses new load_manual_exclusions utility) ──────────────
from scripts.utils import load_manual_exclusions as _lme

_analysis_langs, _search_terms, _corrections = _lme(EVAL_DIR)
print(f"Manual exclusions loaded:")
print(f"  analysis_exclusion : {len(_analysis_langs)} language codes  (dropped from analysis)")
print(f"  search_exclusion   : {len(_search_terms)} (language, term) pairs")
print(f"  term_correction    : {len(_corrections)} corrections")

Manual exclusions loaded:
  analysis_exclusion : 84 language codes  (dropped from analysis)
  search_exclusion   : 209 (language, term) pairs
  term_correction    : 82 corrections


In [5]:
# Canonical run: string features + Wikipedia only, no rationale keyword rules.
# Writes disagreement_analysis_no_keywords.csv, then promoted to the canonical path
# read by notebooks 06 and 07.
cfg = ClassifierConfig(use_keyword_rules=False)

result_df = run_disagreement_analysis(
    data_directory_path=DATA_DIR,
    target_terms=['Digital Humanities'],
    rationale_variant='minimal',
    config=cfg,
    exclusions=tier2_excl,
)

result_df.to_csv(DISAGR_CSV, index=False)
print(f'\nCanonical CSV saved → {DISAGR_CSV}')
print(f'Rows: {len(result_df):,}  |  Columns: {len(result_df.columns)}')

Classifier configuration

rationale_variant        = minimal

rationales_are_english   = True

use_keyword_rules        = False

norm_threshold           = 0.15

...

   Rule-fired counts (which rules did the work)    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ Rule                            ┃ Count ┃ Pct   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ rule_default_transmogrification │ 421   │ 47.8% │
│ rule_loanword_absence           │ 416   │ 47.3% │
│ rule_wikipedia_presence         │ 35    │ 4.0%  │
│ rule_all_agree                  │ 4     │ 0.5%  │
│ rule_normalization_artifact     │ 4     │ 0.5%  │
└─────────────────────────────────┴───────┴───────┘

Category × Language Family

language_family  PRODUCTIVE_DISAGREEMENT  STRUCTURAL_ABSENCE  TRANSMOGRIFICATION
           Afro-Asiatic languages                        2                  10                  39
               Albanian languages                        0                   0                   2
                 Altaic languages                        2                   6                  26
               Armenian languages                        1                   0                   0
             Artificial languages                        0                   4                  10
             Australian languages                        0                   1                   0
         Austro-Asiatic languages                        1                  10                   3
           Austronesian languages                        1                  75                   8
                 Basque languages                        1                   0                   0
              Caucasian languages                        0                   1                  14
Central American Indian languages                        0                   3                   4
    Chukotko-Kamchatkan languages                        0                   0                   2
              Creoles and pidgins                        0                  14                   3
              Dravidian languages                        1                   3                   6
           Eskimo-Aleut languages                        0                   3                   2
             Hmong-Mien languages                        0                   4                   0
          Indo-European languages                       18                  56                 165
                Japonic languages                        1                   0                   1
                Khoisan languages                        0                   1                   0
                 Language isolate                        1                   6                   7
      Niger-Kordofanian languages                        1                 129                  28
           Nilo-Saharan languages                        0                  24                   5
  North American Indian languages                        0                  30                  35
                   Sign languages                        0                   1                   0
           Sino-Tibetan languages                        1                  22                  26
  South American Indian languages                        0                   9                   5
              Tai-Kadai languages                        1                   2                   7
              Undeciphered script                        0                   1                   0
                 Uralic languages                        3                   1                  23

✓ Wrote 880 rows → 
/Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evalu
ation/disagreement_analysis_no_keywords.csv


Canonical CSV saved → /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/disagreement_analysis.csv
Rows: 880  |  Columns: 26


## 5.2 The Consensus Gradient

How many languages fall into each category? The five categories form a gradient from full consensus to maximum uncertainty.

In [6]:
CAT_ORDER = [
    'COMPLETE_CONSENSUS',
    'MEASUREMENT_ARTEFACT',
    'PRODUCTIVE_DISAGREEMENT',
    'STRUCTURAL_ABSENCE',
    'TRANSMOGRIFICATION',
]
CAT_COLORS = {
    'COMPLETE_CONSENSUS':      '#6baed6',
    'MEASUREMENT_ARTEFACT':    '#74c476',
    'PRODUCTIVE_DISAGREEMENT': '#fd8d3c',
    'STRUCTURAL_ABSENCE':      '#9e9ac8',
    'TRANSMOGRIFICATION':      '#de2d26',
}
CAT_LABELS = {
    'COMPLETE_CONSENSUS':      'Complete consensus (all services identical)',
    'MEASUREMENT_ARTEFACT':    'Surface variation (case / diacritics / script)',
    'PRODUCTIVE_DISAGREEMENT': 'Productive disagreement (Wikipedia-anchored)',
    'STRUCTURAL_ABSENCE':      'Structural absence (source-term borrowing)',
    'TRANSMOGRIFICATION':      'Transmogrification (divergent, unanchored)',
}


In [7]:
cat_df = result_df['category'].value_counts().reset_index()
cat_df['label'] = cat_df['category'].map(CAT_LABELS)
cat_df['pct']   = (cat_df['count'] / len(result_df) * 100).round(1)

print('Category distribution:')
for _, r in cat_df.sort_values('count', ascending=False).iterrows():
    bar = '█' * int(r['count'] / len(result_df) * 40)
    print(f"  {r['category']:<30} {bar} {int(r['count']):>4}  ({r['pct']:.1f}%)")

alt.Chart(cat_df).mark_bar().encode(
    x=alt.X('count:Q', title='Number of languages'),
    y=alt.Y('category:N', sort=CAT_ORDER, title=None),
    color=alt.Color(
        'category:N',
        scale=alt.Scale(domain=list(CAT_COLORS.keys()), range=list(CAT_COLORS.values())),
        legend=None,
    ),
    tooltip=[
        alt.Tooltip('label:N', title='Category'),
        alt.Tooltip('count:Q', title='Languages'),
        alt.Tooltip('pct:Q', format='.1f', title='%'),
    ],
).properties(
    width=520, height=220,
    title=alt.TitleParams(
        'Translation disagreement gradient — 880 languages',
        subtitle='Ordered from consensus (top) to maximum uncertainty (bottom)',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)

Category distribution:
  TRANSMOGRIFICATION             ███████████████████  421  (47.8%)
  STRUCTURAL_ABSENCE             ██████████████████  416  (47.3%)
  PRODUCTIVE_DISAGREEMENT        █   35  (4.0%)
  COMPLETE_CONSENSUS                 4  (0.5%)
  MEASUREMENT_ARTEFACT               4  (0.5%)


alt.Chart(...)

## 5.3 Rule Audit

The `rule_fired` column records which rule in the chain produced each verdict. With keyword rules disabled, `rule_sparse_coverage_absence` and `rule_convergent_absence` must show zero counts, confirming the classification rests entirely on string features and Wikipedia.

In [8]:
KEYWORD_RULES = {'rule_sparse_coverage_absence', 'rule_convergent_absence'}

rule_counts = result_df['rule_fired'].value_counts().reset_index()
rule_counts.columns = ['rule', 'n']
rule_counts['pct'] = (rule_counts['n'] / len(result_df) * 100).round(1)
rule_counts['is_keyword'] = rule_counts['rule'].isin(KEYWORD_RULES)

print('Rules that fired (keyword rules disabled — must be 0):')
for _, r in rule_counts.iterrows():
    flag = '  ← KEYWORD RULE (should be 0)' if r['is_keyword'] else ''
    print(f"  {r['rule']:<42} {r['n']:>4}  ({r['pct']:.1f}%){flag}")

alt.Chart(rule_counts).mark_bar().encode(
    x=alt.X('n:Q', title='Languages classified by this rule'),
    y=alt.Y('rule:N', sort='-x', title=None),
    color=alt.condition(
        alt.datum.is_keyword,
        alt.value('#de2d26'),
        alt.value('#4292c6'),
    ),
    tooltip=['rule:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='%')],
).properties(
    width=500, height=240,
    title=alt.TitleParams(
        'Rule-fired audit',
        subtitle='Red = keyword-dependent rules (should be 0 with use_keyword_rules=False)',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)

Rules that fired (keyword rules disabled — must be 0):
  rule_default_transmogrification             421  (47.8%)
  rule_loanword_absence                       416  (47.3%)
  rule_wikipedia_presence                      35  (4.0%)
  rule_all_agree                                4  (0.5%)
  rule_normalization_artifact                   4  (0.5%)


alt.Chart(...)

## 5.4 Keyword Rule Ablation

The classifier has two keyword-dependent rules — `rule_sparse_coverage_absence` and `rule_convergent_absence` — that fire when a rationale contains absence-signalling phrases ("untranslatable", "no equivalent", etc.). The canonical run above deliberately disables them to keep the classification grounded in string features alone, ensuring the disagreement labels remain independent of the rationale analysis in notebook 07.

This section re-runs the classifier with keyword rules enabled and compares the two outputs. The question: do keyword rules meaningfully shift the category distribution, or are they redundant with the string-feature rules?

If the shift is small (< 5% of languages change category), the no-keyword canonical is well-justified. If large, keyword rules carry independent signal worth understanding before deciding on a canonical.

In [9]:
cfg_kw = ClassifierConfig(use_keyword_rules=True)
kw_result_df = run_disagreement_analysis(
    data_directory_path=DATA_DIR,
    target_terms=['Digital Humanities'],
    rationale_variant='minimal',
    exclusions=tier2_excl,
    config=cfg_kw,
)
print(f'\nKeyword-enabled CSV: {DISAGR_KW_CSV}')

Classifier configuration

rationale_variant        = minimal

rationales_are_english   = True

use_keyword_rules        = True

norm_threshold           = 0.15

...

   Rule-fired counts (which rules did the work)    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ Rule                            ┃ Count ┃ Pct   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ rule_default_transmogrification │ 421   │ 47.8% │
│ rule_loanword_absence           │ 416   │ 47.3% │
│ rule_wikipedia_presence         │ 35    │ 4.0%  │
│ rule_all_agree                  │ 4     │ 0.5%  │
│ rule_normalization_artifact     │ 4     │ 0.5%  │
└─────────────────────────────────┴───────┴───────┘

Category × Language Family

language_family  PRODUCTIVE_DISAGREEMENT  STRUCTURAL_ABSENCE  TRANSMOGRIFICATION
           Afro-Asiatic languages                        2                  10                  39
               Albanian languages                        0                   0                   2
                 Altaic languages                        2                   6                  26
               Armenian languages                        1                   0                   0
             Artificial languages                        0                   4                  10
             Australian languages                        0                   1                   0
         Austro-Asiatic languages                        1                  10                   3
           Austronesian languages                        1                  75                   8
                 Basque languages                        1                   0                   0
              Caucasian languages                        0                   1                  14
Central American Indian languages                        0                   3                   4
    Chukotko-Kamchatkan languages                        0                   0                   2
              Creoles and pidgins                        0                  14                   3
              Dravidian languages                        1                   3                   6
           Eskimo-Aleut languages                        0                   3                   2
             Hmong-Mien languages                        0                   4                   0
          Indo-European languages                       18                  56                 165
                Japonic languages                        1                   0                   1
                Khoisan languages                        0                   1                   0
                 Language isolate                        1                   6                   7
      Niger-Kordofanian languages                        1                 129                  28
           Nilo-Saharan languages                        0                  24                   5
  North American Indian languages                        0                  30                  35
                   Sign languages                        0                   1                   0
           Sino-Tibetan languages                        1                  22                  26
  South American Indian languages                        0                   9                   5
              Tai-Kadai languages                        1                   2                   7
              Undeciphered script                        0                   1                   0
                 Uralic languages                        3                   1                  23

✓ Wrote 880 rows → 
/Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evalu
ation/disagreement_analysis.csv


Keyword-enabled CSV: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/disagreement_analysis_keywords.csv


In [10]:
no_kw_slim = pd.read_csv(DISAGR_NO_KW_CSV)[
    ['language_code', 'language_name', 'language_family', 'category', 'rule_fired']
].rename(columns={'category': 'cat_no_kw', 'rule_fired': 'rule_no_kw'})

kw_slim = pd.read_csv(DISAGR_KW_CSV)[
    ['language_code', 'category', 'rule_fired']
].rename(columns={'category': 'cat_kw', 'rule_fired': 'rule_kw'})

ablation_df = no_kw_slim.merge(kw_slim, on='language_code')
ablation_df['changed'] = ablation_df['cat_no_kw'] != ablation_df['cat_kw']

print(f'Total languages: {len(ablation_df)}')
print(f'Changed category when keywords enabled: {ablation_df["changed"].sum()} ({ablation_df["changed"].mean()*100:.1f}%)')
print()
print('Flow (no-kw → with-kw) for languages that change:')
flow = (
    ablation_df[ablation_df['changed']]
    .groupby(['cat_no_kw', 'cat_kw'])
    .size()
    .reset_index(name='n')
)
print(flow.to_string(index=False) if len(flow) else '  (none)')

Total languages: 856
Changed category when keywords enabled: 252 (29.4%)

Flow (no-kw → with-kw) for languages that change:
              cat_no_kw                  cat_kw   n
   MEASUREMENT_ARTEFACT      COMPLETE_CONSENSUS   1
   MEASUREMENT_ARTEFACT PRODUCTIVE_DISAGREEMENT   1
PRODUCTIVE_DISAGREEMENT      COMPLETE_CONSENSUS   3
PRODUCTIVE_DISAGREEMENT    MEASUREMENT_ARTEFACT   1
     STRUCTURAL_ABSENCE    MEASUREMENT_ARTEFACT   5
     STRUCTURAL_ABSENCE      TRANSMOGRIFICATION 156
     TRANSMOGRIFICATION      COMPLETE_CONSENSUS   5
     TRANSMOGRIFICATION    MEASUREMENT_ARTEFACT   6
     TRANSMOGRIFICATION      STRUCTURAL_ABSENCE  74


In [11]:
no_kw_counts = (
    ablation_df.groupby('cat_no_kw').size()
    .reset_index(name='n')
    .rename(columns={'cat_no_kw': 'category'})
    .assign(run='No keywords (canonical)')
)
kw_counts = (
    ablation_df.groupby('cat_kw').size()
    .reset_index(name='n')
    .rename(columns={'cat_kw': 'category'})
    .assign(run='With keywords')
)
compare_df = pd.concat([no_kw_counts, kw_counts])
compare_df['pct'] = (
    compare_df.groupby('run')['n']
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
)

RUN_ORDER = ['No keywords (canonical)', 'With keywords']

alt.Chart(compare_df).mark_bar().encode(
    x=alt.X('category:N', sort=CAT_ORDER, title=None, axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('n:Q', title='Languages'),
    color=alt.Color(
        'run:N',
        sort=RUN_ORDER,
        scale=alt.Scale(domain=RUN_ORDER, range=['#4292c6', '#de2d26']),
        title='Run',
    ),
    xOffset=alt.XOffset('run:N', sort=RUN_ORDER),
    tooltip=['category:N', 'run:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='%')],
).properties(
    width=540, height=300,
    title='Category distribution: no-keywords (canonical) vs. keyword-enabled',
)

alt.Chart(...)

In [12]:
flipped = (
    ablation_df[ablation_df['changed']]
    [['language_code', 'language_name', 'language_family', 'cat_no_kw', 'rule_no_kw', 'cat_kw', 'rule_kw']]
    .rename(columns={
        'cat_no_kw':  'category (no kw)',
        'rule_no_kw': 'rule (no kw)',
        'cat_kw':     'category (with kw)',
        'rule_kw':    'rule (with kw)',
    })
    .sort_values('category (with kw)')
)

n_flipped   = len(flipped)
pct_flipped = n_flipped / len(ablation_df) * 100

print(f'Languages that change category: {n_flipped} / {len(ablation_df)} ({pct_flipped:.1f}%)')
print()
if n_flipped == 0:
    print('Verdict: keyword rules add no independent signal — no-keyword canonical is fully justified.')
elif pct_flipped < 5:
    print(f'Verdict: keyword rules are somewhat marginal ({pct_flipped:.1f}% shift). No-keyword canonical is generally well-justified, though shift is still worth investigating.')
else:
    print(f'Verdict: keyword rules carry independent signal ({pct_flipped:.1f}% shift). Review flipped cases before finalising canonical.')

if n_flipped > 0:
    print()
    print(flipped.to_string(index=False))

Languages that change category: 252 / 856 (29.4%)

Verdict: keyword rules carry independent signal (29.4% shift). Review flipped cases before finalising canonical.

language_code               language_name                   language_family        category (no kw)                    rule (no kw)      category (with kw)                  rule (with kw)
          ars                Najdi Arabic            Afro-Asiatic languages      TRANSMOGRIFICATION rule_default_transmogrification      COMPLETE_CONSENSUS                  rule_all_agree
           el               Greek, Modern           Indo-European languages PRODUCTIVE_DISAGREEMENT         rule_wikipedia_presence      COMPLETE_CONSENSUS                  rule_all_agree
           fa                     Persian           Indo-European languages PRODUCTIVE_DISAGREEMENT         rule_wikipedia_presence      COMPLETE_CONSENSUS                  rule_all_agree
           ia                 Interlingua              Artificial languages    MEAS

## 5.5 String Features per Category

The raw counts that drove the classification, including unique output counts and Levenshtein edit distances between normalised strings. These are the features the classifier actually read; there is no other signal.

In [13]:
feat_df = result_df[['category', 'n_unique_raw', 'n_unique_normalized', 'max_edit_distance']].copy()
feat_df['max_edit_distance'] = pd.to_numeric(feat_df['max_edit_distance'], errors='coerce')

print('Mean string features per category:')
print(
    feat_df.groupby('category')[['n_unique_raw', 'n_unique_normalized', 'max_edit_distance']]
    .mean().round(2).reindex(CAT_ORDER).to_string()
)

strip_chart = alt.Chart(feat_df).mark_tick(
    thickness=1.5, opacity=0.35,
).encode(
    x=alt.X('n_unique_raw:Q', title='N unique raw translations'),
    y=alt.Y('category:N', sort=CAT_ORDER, title=None),
    color=alt.Color(
        'category:N',
        scale=alt.Scale(domain=list(CAT_COLORS.keys()), range=list(CAT_COLORS.values())),
        legend=None,
    ),
    tooltip=['category:N', 'n_unique_raw:Q', 'max_edit_distance:Q'],
).properties(width=460, height=200, title='Unique raw translation count per category')

box_chart = alt.Chart(
    feat_df[
        feat_df['max_edit_distance'].notna() &
        (feat_df['max_edit_distance'] > 0)
    ]
).mark_boxplot(extent=1.5).encode(
    x=alt.X('max_edit_distance:Q', title='Max pairwise Levenshtein distance'),
    y=alt.Y('category:N', sort=CAT_ORDER, title=None),
    color=alt.Color(
        'category:N',
        scale=alt.Scale(domain=list(CAT_COLORS.keys()), range=list(CAT_COLORS.values())),
        legend=None,
    ),
).properties(
    width=460, height=200,
    title='Edit distance per category (zero-distance rows excluded)',
)

alt.vconcat(strip_chart, box_chart).resolve_scale(color='shared')

Mean string features per category:
                         n_unique_raw  n_unique_normalized  max_edit_distance
category                                                                     
COMPLETE_CONSENSUS               1.00                 1.00               0.00
MEASUREMENT_ARTEFACT             3.25                 3.25               2.00
PRODUCTIVE_DISAGREEMENT          5.34                 5.29              20.40
STRUCTURAL_ABSENCE               7.41                 7.36              26.29
TRANSMOGRIFICATION               6.97                 6.91              23.84


alt.VConcatChart(...)

## 5.6 Category × Language Family

Does the disagreement pattern vary systematically by language family? Rows are sorted by TRANSMOGRIFICATION rate (highest first) to surface which families are most susceptible to fabricated output.

In [14]:
FAMILY_SHORT = {
    'Afro-Asiatic languages':            'Afro-Asiatic',
    'Altaic languages':                   'Altaic',
    'Artificial languages':               'Artificial',
    'Austro-Asiatic languages':           'Austro-Asiatic',
    'Austronesian languages':             'Austronesian',
    'Caucasian languages':                'Caucasian',
    'Central American Indian languages':  'C. American Indian',
    'Creoles and pidgins':                'Creoles/Pidgins',
    'Dravidian languages':                'Dravidian',
    'Eskimo-Aleut languages':             'Eskimo-Aleut',
    'Hmong-Mien languages':               'Hmong-Mien',
    'Indo-European languages':            'Indo-European',
    'Language isolate':                   'Isolates',
    'Niger-Kordofanian languages':        'Niger-Kordofanian',
    'Nilo-Saharan languages':             'Nilo-Saharan',
    'North American Indian languages':    'N. American Indian',
    'Sino-Tibetan languages':             'Sino-Tibetan',
    'South American Indian languages':    'S. American Indian',
    'Tai languages':                      'Tai',
    'Uralic languages':                   'Uralic',
}

# Restrict to families with at least 5 languages for readability
family_sizes = result_df['language_family'].value_counts()
major_families = family_sizes[family_sizes >= 5].index
plot_df = result_df[result_df['language_family'].isin(major_families)].copy()
plot_df['family_short'] = plot_df['language_family'].map(FAMILY_SHORT).fillna(plot_df['language_family'])

family_cat = (
    plot_df.groupby(['family_short', 'category'])
    .size()
    .reset_index(name='n')
)
totals = family_cat.groupby('family_short')['n'].sum().rename('total')
family_cat = family_cat.join(totals, on='family_short')
family_cat['pct'] = (family_cat['n'] / family_cat['total'] * 100).round(1)

# Sort families by TRANSMOGRIFICATION rate
tmog_pct = (
    family_cat[family_cat['category'] == 'TRANSMOGRIFICATION']
    .set_index('family_short')['pct']
)
family_order = list(tmog_pct.sort_values(ascending=False).index)

print('TRANSMOGRIFICATION rate by family (top 10):')
for fam in family_order[:10]:
    n_total = int(totals.get(fam, 0))
    pct = tmog_pct.get(fam, 0)
    print(f'  {fam:<22} {pct:.1f}%  (n={n_total})')

alt.Chart(family_cat).mark_rect().encode(
    x=alt.X('category:N', sort=CAT_ORDER, title=None,
            axis=alt.Axis(labelAngle=-30, labelLimit=220)),
    y=alt.Y('family_short:N', sort=family_order, title=None),
    color=alt.Color('pct:Q', scale=alt.Scale(scheme='blues'), title='% of family'),
    tooltip=[
        'family_short:N', 'category:N',
        alt.Tooltip('n:Q', title='Languages'),
        alt.Tooltip('pct:Q', format='.1f', title='% of family'),
    ],
).properties(
    width=520, height=380,
    title=alt.TitleParams(
        'Disagreement category × language family',
        subtitle='% of each family in each category (families sorted by TRANSMOGRIFICATION rate)',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)

TRANSMOGRIFICATION rate by family (top 10):
  Caucasian              93.3%  (n=15)
  Uralic                 85.2%  (n=27)
  Altaic                 76.5%  (n=34)
  Afro-Asiatic           76.5%  (n=51)
  Tai-Kadai languages    70.0%  (n=10)
  Indo-European          67.6%  (n=244)
  Artificial             66.7%  (n=15)
  Dravidian              60.0%  (n=10)
  C. American Indian     57.1%  (n=7)
  N. American Indian     53.8%  (n=65)


alt.Chart(...)

## 5.7 STRUCTURAL_ABSENCE — Loan-Word Detail

These languages had at least one service output the source term unadapted. The `loan_words_found` column records which tokens were matched. Source tokens are derived automatically from the translation term, so for "Digital Humanities" we check for "digital", "humanities", "dh", and "digital humanities" as a compound. This is a crude heuristic but surfaces interesting cases of apparent borrowing worth investigating further with native speakers.

In [15]:
absence_df = result_df[result_df['category'] == 'STRUCTURAL_ABSENCE'].copy()
print(f'STRUCTURAL_ABSENCE: {len(absence_df)} languages')

# Parse semicolon-separated loan_words_found
token_rows = []
for _, row in absence_df.iterrows():
    raw = str(row.get('loan_words_found', '') or '')
    if raw.strip() and raw not in ('nan', 'None'):
        for tok in raw.split(';'):
            tok = tok.strip()
            if tok:
                token_rows.append({
                    'language_code':   row['language_code'],
                    'language_name':   row['language_name'],
                    'language_family': row['language_family'],
                    'token': tok,
                })

if token_rows:
    token_df   = pd.DataFrame(token_rows)
    tok_counts = token_df['token'].value_counts().reset_index()
    tok_counts.columns = ['token', 'n']

    print(f'Languages with detected loan words: {token_df["language_code"].nunique()}')
    print(f'\nToken frequency:')
    print(tok_counts.to_string(index=False))

    alt.Chart(tok_counts).mark_bar(color='#9e9ac8').encode(
        x=alt.X('n:Q', title='Languages borrowing this token'),
        y=alt.Y('token:N', sort='-x', title=None),
        tooltip=['token:N', 'n:Q'],
    ).properties(
        width=400, height=200,
        title='Source-language tokens borrowed unadapted in translation outputs',
    )
else:
    print('No loan-word tokens detected in this run.')
    print('All STRUCTURAL_ABSENCE cases were classified by rule_no_translations.')
    print('Check SOURCE_TOKENS in explore_disagreements.py if unexpected.')

STRUCTURAL_ABSENCE: 416 languages
Languages with detected loan words: 416

Token frequency:
     token   n
   digital 404
humanities 255


## 5.8 TRANSMOGRIFICATION — Characterising Maximum Uncertainty

The default category: multiple distinct outputs, no Wikipedia anchor, no detectable source-term borrowing. These are the most methodologically interesting cases — services produced word-like objects that cannot be verified against community practice.

In [16]:
tmog_df = result_df[result_df['category'] == 'TRANSMOGRIFICATION'].copy()
tmog_df['max_edit_distance'] = pd.to_numeric(tmog_df['max_edit_distance'], errors='coerce')

print(f'TRANSMOGRIFICATION: {len(tmog_df)} languages ({len(tmog_df)/len(result_df)*100:.1f}%)')
print(f'\nn_unique_raw distribution:')
print(tmog_df['n_unique_raw'].describe().round(2).to_string())
print(f'\nWikipedia present in any TRANSMOGRIFICATION case: {tmog_df["has_wikipedia"].any()}')

print(f'\nTop 20 most-divergent cases (by n_unique_raw):')
sample_cols = ['language_code', 'language_name', 'language_family', 'n_unique_raw', 'max_edit_distance']
print(tmog_df.nlargest(20, 'n_unique_raw')[sample_cols].to_string(index=False))

hist = alt.Chart(tmog_df).mark_bar(color='#de2d26', opacity=0.8).encode(
    x=alt.X('n_unique_raw:Q', bin=alt.Bin(maxbins=10),
            title='N unique raw translations'),
    y=alt.Y('count()', title='Languages'),
    tooltip=[alt.Tooltip('n_unique_raw:Q', bin=True), 'count()'],
).properties(
    width=420, height=220,
    title='Output diversity within TRANSMOGRIFICATION',
)

sim_chart = alt.Chart(
    tmog_df.dropna(subset=['mean_rationale_similarity'])
).mark_point(opacity=0.4, size=30, color='#de2d26').encode(
    x=alt.X('n_unique_raw:Q', title='N unique raw translations'),
    y=alt.Y('mean_rationale_similarity:Q', title='Mean rationale similarity'),
    tooltip=['language_name:N', 'n_unique_raw:Q',
             alt.Tooltip('mean_rationale_similarity:Q', format='.3f')],
).properties(
    width=420, height=220,
    title=alt.TitleParams(
        'Translation divergence vs. rationale similarity (TRANSMOGRIFICATION)',
        subtitle='Preview of notebook 07 two-axis analysis',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)

alt.hconcat(hist, sim_chart)

TRANSMOGRIFICATION: 421 languages (47.8%)

n_unique_raw distribution:
count    421.00
mean       6.97
std        1.51
min        3.00
25%        6.00
50%        7.00
75%        8.00
max       10.00

Wikipedia present in any TRANSMOGRIFICATION case: False

Top 20 most-divergent cases (by n_unique_raw):
language_code         language_name             language_family  n_unique_raw  max_edit_distance
          crs Seselwa Creole French         Creoles and pidgins            10                 29
           fj                Fijian      Austronesian languages            10                 35
           gv                  Manx     Indo-European languages            10                 22
           ht               Haitian     Indo-European languages            10                 28
          lua            Luba-Lulua Niger-Kordofanian languages            10                 29
           om                 Oromo      Afro-Asiatic languages            10                 25
           so     

alt.HConcatChart(...)

## 5.9 Rebuild Downstream Data

The updated `disagreement_analysis.csv` must be propagated to the downstream data layer before running notebooks 06 or 07.

This step runs `build_disagreement_explorer_data.py`, which merges disagreement analysis, per-language confidence stats, and parsed service translations into `disagreement_explorer_data.csv`. That CSV is consumed by notebook 07 (rationale classification) — it is **not** an HTML explorer feed.

In [17]:
import subprocess

repo_root = str(Path('..').resolve())
proc = subprocess.run(
    ['python3', 'scripts/exploration/build_disagreement_explorer_data.py'],
    capture_output=True, text=True, cwd=repo_root,
)
print(proc.stdout)
if proc.returncode != 0:
    print('STDERR:', proc.stderr[:800])

Retrieving translation pipeline data directory path...
Saved 880 rows × 109 cols → /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/disagreement_explorer_data.csv
  TRANSMOGRIFICATION: 421
  STRUCTURAL_ABSENCE: 416
  PRODUCTIVE_DISAGREEMENT: 35
  MEASUREMENT_ARTEFACT: 4
  COMPLETE_CONSENSUS: 4

